# Human-in-the-Loop & Approvals

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to require human approval for sensitive tool calls using `needs_approval=True`, inspect interruptions, convert to `RunState`, and approve or reject pending calls before resuming.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define Tools That Need Approval

Pass `needs_approval=True` to `function_tool` so the agent pauses instead of executing the tool directly.

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool(needs_approval=True)
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to the specified recipient."""
    return f"Email sent to {to} with subject '{subject}'"


@function_tool(needs_approval=True)
def delete_record(record_id: str) -> str:
    """Permanently delete a record from the database."""
    return f"Record {record_id} deleted"


agent = Agent(
    name="Assistant",
    instructions="You help users manage their emails and records.",
    tools=[send_email, delete_record],
)

print("Agent created with approval-required tools.")

## 4. Run and Check Interruptions

When the agent wants to call an approval-required tool, the run returns with interruptions instead of a final output.

In [ ]:
result = await Runner.run(agent, "Send an email to alice@example.com about the meeting tomorrow")

if result.interruptions:
    for interruption in result.interruptions:
        print(f"Tool: {interruption.tool_name}")
        print(f"Arguments: {interruption.tool_arguments}")
else:
    print(result.final_output)

## 5. Approve and Resume

Convert the result to a `RunState`, approve the pending call, then resume execution.

In [ ]:
state = result.to_state()

state.approve(interruption_id=result.interruptions[0].id)

resumed_result = await Runner.run(agent, state)
print(resumed_result.final_output)

## 6. Reject a Tool Call

Reject the pending call with a message. The agent receives the rejection reason and can adjust its response.

In [ ]:
result = await Runner.run(agent, "Delete record DR-42 from the database")

state = result.to_state()

state.reject(
    interruption_id=result.interruptions[0].id,
    message="Not authorized to delete production records",
)

resumed_result = await Runner.run(agent, state)
print(resumed_result.final_output)

## 7. Handle Multiple Pending Approvals

When the agent triggers multiple approval-required tools, approve or reject each independently.

In [ ]:
result = await Runner.run(agent, "Send the report to alice@example.com and delete the draft record DR-42")

state = result.to_state()

for interruption in result.interruptions:
    if interruption.tool_name == "send_email":
        state.approve(interruption_id=interruption.id)
    elif interruption.tool_name == "delete_record":
        state.reject(
            interruption_id=interruption.id,
            message="Deletion requires manager approval",
        )

resumed_result = await Runner.run(agent, state)
print(resumed_result.final_output)

## 8. Custom Rejection Messages with tool_error_formatter

Use `tool_error_formatter` to customize the message the agent sees when a tool call is rejected.

In [ ]:
def format_rejection(tool_name: str, error: str) -> str:
    return (
        f"The tool '{tool_name}' was rejected by a human reviewer. "
        f"Reason: {error}. Please inform the user and suggest alternatives."
    )


careful_agent = Agent(
    name="Careful Assistant",
    instructions="You help users manage sensitive operations.",
    tools=[send_email, delete_record],
    tool_error_formatter=format_rejection,
)

result = await Runner.run(careful_agent, "Delete record DR-99")

state = result.to_state()
state.reject(
    interruption_id=result.interruptions[0].id,
    message="Record is locked by compliance team",
)

resumed_result = await Runner.run(careful_agent, state)
print(resumed_result.final_output)

## Key Takeaways

- Use `function_tool(needs_approval=True)` to require human sign-off before tool execution
- Check `result.interruptions` for pending approval requests after a run
- Convert results to a serializable `RunState` with `result.to_state()`
- Use `state.approve()` and `state.reject()` to authorize or deny tool calls
- Resume execution with `Runner.run(agent, state)` after making decisions
- Customize rejection messages with `tool_error_formatter`